# Hyperparameter Tuning Tutorial



HYPERPARAMETER TUNING

In [ ]:
GRID SEARCH 

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------
# STEP 1: Define the model
# --------------------------------------------------
# This is the machine learning algorithm we want to tune.
# Here we are using Random Forest for classification.
# random_state=42 is used so results are reproducible.
# That means if you run it again, you are more likely to
# get the same random behavior.
model = RandomForestClassifier(random_state=42)

# --------------------------------------------------
# STEP 2: Define the hyperparameter grid
# --------------------------------------------------
# param_grid is a dictionary.
# Each key is a hyperparameter name from the model.
# Each value is a list of candidate values to try.
#
# GridSearchCV will test ALL possible combinations.
#
# Example:
# if n_estimators has 3 values
# and max_depth has 3 values
# and min_samples_split has 2 values
# total combinations = 3 × 3 × 2 = 18
param_grid = {
    'n_estimators': [100, 200, 300],      # number of trees in the forest
    'max_depth': [None, 5, 10],           # maximum depth of each tree
    'min_samples_split': [2, 5, 10]       # minimum samples needed to split a node
}

# --------------------------------------------------
# STEP 3: Create GridSearchCV object
# --------------------------------------------------
grid_search = GridSearchCV(
    estimator=model,          # the model/algorithm we want to tune
    param_grid=param_grid,    # the hyperparameter combinations to test

    scoring='accuracy',       # metric used to decide which combination is best
                              # accuracy = correct predictions / all predictions
                              # good when classes are balanced
                              # not always good for imbalanced datasets

    cv=5,                     # 5-fold cross-validation
                              # data is split into 5 parts
                              # train on 4 parts, validate on 1 part
                              # repeat 5 times so each part is used once for validation
                              # final score = average of those 5 validation scores

    n_jobs=-1,                # number of CPU cores to use
                              # n_jobs=-1 means use all available CPU cores
                              # n_jobs=1 means use only one CPU core
                              # n_jobs=2 means use two CPU cores
                              # faster with -1, especially for large searches

    verbose=1,                # controls how much progress is printed
                              # verbose=0 -> prints nothing
                              # verbose=1 -> basic progress messages
                              # verbose=2 or 3 -> more detailed messages

    return_train_score=True   # also stores training scores
                              # useful for checking possible overfitting
)

# --------------------------------------------------
# STEP 4: Fit on training data
# --------------------------------------------------
# This runs the full search.
# For each hyperparameter combination:
#   1. model is trained
#   2. validated with cross-validation
#   3. score is calculated
# After all combinations are tested,
# the best one is selected.
grid_search.fit(X_train, y_train)

# --------------------------------------------------
# STEP 5: Show best results
# --------------------------------------------------
print("Best parameters:", grid_search.best_params_)   # best hyperparameter combination
print("Best CV score:", grid_search.best_score_)      # best average cross-validation score

# --------------------------------------------------
# STEP 6: Get the best trained model
# --------------------------------------------------
# best_estimator_ gives the model with the best hyperparameters.
best_model = grid_search.best_estimator_

# Now you can use this tuned model on test data.
y_pred = best_model.predict(X_test) 

```

---

# Important arguments explained properly

## `estimator=model`

This is the actual machine learning algorithm you want to tune. It can be `LogisticRegression()`, `RandomForestClassifier()`, `DecisionTreeClassifier()`, `SVC()`, and so on.

So this part answers the question: **which model are we tuning?**

---

## `param_grid=param_grid`

This is the set of hyperparameter values to test.

For example:

```python
param_grid = {
    'max_depth': [5, 10],
    'min_samples_split': [2, 4]
}
```

This means GridSearchCV will test:

* `max_depth=5`, `min_samples_split=2`
* `max_depth=5`, `min_samples_split=4`
* `max_depth=10`, `min_samples_split=2`
* `max_depth=10`, `min_samples_split=4`

So grid search is called “grid” because it checks the full grid of combinations.

---

## `scoring='accuracy'`

This tells sklearn **how to judge model performance**.

### `accuracy`

Accuracy means:

[
\text{Accuracy} = \frac{\text{correct predictions}}{\text{total predictions}}
]

Use accuracy when:

* classes are roughly balanced
* false positives and false negatives are similarly important
* you want overall correctness

Do **not** rely only on accuracy when data is imbalanced.

Example: if 99% of cases are class 0, a dumb model predicting only 0 gets 99% accuracy, but it is actually useless.

---

## Other scoring options and when to use them

### `scoring='precision'`

Precision means:

[
\text{Precision} = \frac{TP}{TP+FP}
]

It answers: **out of predicted positives, how many were actually positive?**

Use precision when:

* false positives are costly
* you want predictions of the positive class to be trustworthy

Example:

* spam detection
* fraud flagging where false alarms are expensive

---

### `scoring='recall'`

Recall means:

[
\text{Recall} = \frac{TP}{TP+FN}
]

It answers: **out of actual positives, how many did we correctly find?**

Use recall when:

* missing a positive case is dangerous
* false negatives are worse than false positives

Example:

* disease detection
* cancer screening
* fraud detection if missing fraud is very bad

This is why in medical classification people often optimize recall.

---

### `scoring='f1'`

F1-score balances precision and recall:

[
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
]

Use F1 when:

* classes are imbalanced
* both precision and recall matter
* you want a compromise between the two

---

### `scoring='roc_auc'`

ROC-AUC measures how well the model separates classes across thresholds.

Use it when:

* you care about ranking positives above negatives
* you want threshold-independent evaluation
* you are working with classification probabilities

Often useful for binary classification.

---

## `cv=5`

This means **5-fold cross-validation**.

Suppose you have training data. Instead of using just one train-validation split, sklearn splits the training data into 5 parts.

Then it does:

* Fold 1: train on 4 parts, validate on 1
* Fold 2: train on 4 parts, validate on another 1
* Fold 3: same idea
* Fold 4: same idea
* Fold 5: same idea

Then it averages the 5 validation scores.

Why do this?
Because performance from one split may be lucky or unlucky. Cross-validation gives a more reliable estimate.

### Typical choices

* `cv=3`: faster, less stable
* `cv=5`: common balance
* `cv=10`: more thorough, slower

---

## `n_jobs=-1`

This controls **how many CPU cores** sklearn can use.

* `n_jobs=1` means use one CPU core only
* `n_jobs=2` means use two CPU cores
* `n_jobs=-1` means use **all available CPU cores**

Why useful?
Grid search can be slow because it trains many models. Parallel processing makes it faster.

Use `-1` when:

* your machine can handle it
* you want faster tuning

Use `1` when:

* you want minimal CPU usage
* you have compatibility/resource issues

---

## `verbose=1`

This controls how much progress information is printed while the search runs.

* `verbose=0` means no printed messages
* `verbose=1` means some progress messages
* `verbose=2` or higher means more detailed messages

Example output:

```python
Fitting 5 folds for each of 12 candidates, totalling 60 fits
```

That means:

* 12 hyperparameter combinations
* each checked with 5-fold CV
* total model fits = 12 × 5 = 60

This is useful because it helps you understand why tuning takes time.

---

## `return_train_score=True`

This stores training scores in addition to validation scores.

Why useful?
Because if training score is very high but validation score is much lower, the model may be overfitting.

Not strictly required, but helpful for analysis.

---

# `param_grid` vs `param_distributions`

## `param_grid`

Used with **GridSearchCV**.

You give exact lists of values, and sklearn tests **every combination**.

Example:

```python
param_grid = {
    'max_depth': [5, 10],
    'min_samples_split': [2, 4]
}
```

All combinations are tested.

---

## `param_distributions`

Used with **RandomizedSearchCV**.

You give a search space, and sklearn randomly samples a fixed number of combinations.

Example:

```python
param_dist = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 4, 6, 8]
}
```

If `n_iter=5`, it tests only 5 random combinations, not all.

Use RandomizedSearchCV when:

* the search space is large
* full grid search would take too long
* you want a faster approximate search

---

# Different models use different hyperparameters

This part is very important: **not every model accepts the same hyperparameters**.

You must use parameter names that belong to that specific algorithm.

---

## 1. Logistic Regression

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

param_grid = {
    'C': [0.01, 0.1, 1, 10],     # inverse of regularization strength
    'penalty': ['l2'],           # regularization type
    'solver': ['lbfgs', 'liblinear']
}
```

### Meaning

* `C`: smaller `C` means stronger regularization, larger `C` means weaker regularization
* `penalty`: type of regularization
* `solver`: optimization algorithm used to fit the model

---

## 2. Decision Tree

```python
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)

param_grid = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}
```

### Meaning

* `max_depth`: maximum tree depth
* `min_samples_split`: minimum number of samples needed to split a node
* `min_samples_leaf`: minimum number of samples in a leaf node
* `criterion`: how split quality is measured

---

## 3. Random Forest

```python
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
```

### Meaning

* `n_estimators`: number of trees
* `max_depth`: depth of each tree
* `min_samples_split`: minimum samples to split
* `min_samples_leaf`: minimum samples in a leaf
* `max_features`: number of features considered at each split

---

## 4. SVM

```python
from sklearn.svm import SVC

model = SVC()

param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3]   # only relevant if kernel='poly'
}
```

### Meaning

* `C`: regularization strength control
* `kernel`: transformation type used
* `gamma`: influence range of each training point
* `degree`: polynomial degree, only for polynomial kernel

Important:
some parameters matter only for specific kernels. For example `degree` is relevant mainly for `poly`.

---

## 5. KNN

```python
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()

param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}
```

### Meaning

* `n_neighbors`: number of nearby points used
* `weights`: whether neighbors vote equally or closer ones matter more
* `metric`: distance formula

---

# Very reusable commented template

```python
from sklearn.model_selection import GridSearchCV

# Put your model here
model = SomeModel()

# Put hyperparameters for THAT model here
param_grid = {
    'parameter_name_1': [value1, value2, value3],
    'parameter_name_2': [value1, value2]
}

grid_search = GridSearchCV(
    estimator=model,          # the algorithm we want to tune
    param_grid=param_grid,    # the hyperparameter combinations to test
    scoring='accuracy',       # evaluation metric
    cv=5,                     # number of cross-validation folds
    n_jobs=-1,                # use all CPU cores
    verbose=1                 # print progress
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

best_model = grid_search.best_estimator_
```

---

Written by Chat GPT Model 5.4